# QLoRA Fine-tuning: SQL Expert LLM

Fine-tune Llama 3.1 8B for SQL/Database expertise using QLoRA.

**Requirements:**
- Google Colab with T4 GPU (free tier works!)
- Hugging Face account (for model access)

**Training Time:** ~45 minutes on T4 GPU

## 1. Setup & Installation

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
%%capture
# Install dependencies
!pip install -U transformers datasets accelerate peft bitsandbytes trl
!pip install -U huggingface_hub
!pip install scipy

In [ ]:
# Login to Hugging Face (required for Llama access)
from huggingface_hub import login

# Get your token from: https://huggingface.co/settings/tokens
# Make sure you've accepted Llama license at: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
login()

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# Model configuration
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
NEW_MODEL_NAME = "llama-3.1-8b-sql-expert"

# QLoRA configuration
LORA_R = 16                # LoRA rank (capacity)
LORA_ALPHA = 32            # LoRA scaling factor
LORA_DROPOUT = 0.05        # Dropout for regularization

# Training configuration
EPOCHS = 3
BATCH_SIZE = 2             # Per device (small for T4)
GRADIENT_ACCUMULATION = 4  # Effective batch = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512
WARMUP_RATIO = 0.03

# Output directory
OUTPUT_DIR = "./outputs"

## 3. Prepare Dataset

We'll create a SQL Q&A dataset for fine-tuning.

In [ ]:
# SQL Q&A Training Data
# In production, you'd load from a file or Hugging Face datasets

SQL_TRAINING_DATA = [
    # Basic SELECT queries
    {
        "instruction": "Write a SQL query to select all columns from the employees table.",
        "input": "",
        "output": "SELECT * FROM employees;"
    },
    {
        "instruction": "Write a SQL query to find all employees with salary greater than 50000.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "SELECT * FROM employees WHERE salary > 50000;"
    },
    {
        "instruction": "Write a SQL query to count the number of employees in each department.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "SELECT department, COUNT(*) as employee_count\nFROM employees\nGROUP BY department;"
    },
    # JOIN queries
    {
        "instruction": "Write a SQL query to find all orders with customer names.",
        "input": "Tables: customers(id, name, email), orders(id, customer_id, amount, order_date)",
        "output": "SELECT o.id, c.name, o.amount, o.order_date\nFROM orders o\nJOIN customers c ON o.customer_id = c.id;"
    },
    {
        "instruction": "Write a SQL query to find the top 5 customers by total order value.",
        "input": "Tables: customers(id, name), orders(id, customer_id, amount)",
        "output": "SELECT c.name, SUM(o.amount) as total_value\nFROM customers c\nJOIN orders o ON c.id = o.customer_id\nGROUP BY c.id, c.name\nORDER BY total_value DESC\nLIMIT 5;"
    },
    {
        "instruction": "Write a SQL query to find customers who have never placed an order.",
        "input": "Tables: customers(id, name), orders(id, customer_id, amount)",
        "output": "SELECT c.name\nFROM customers c\nLEFT JOIN orders o ON c.id = o.customer_id\nWHERE o.id IS NULL;"
    },
    # Aggregation queries
    {
        "instruction": "Write a SQL query to calculate the average, min, and max salary by department.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "SELECT department,\n       AVG(salary) as avg_salary,\n       MIN(salary) as min_salary,\n       MAX(salary) as max_salary\nFROM employees\nGROUP BY department;"
    },
    {
        "instruction": "Write a SQL query to find departments with more than 10 employees and average salary above 60000.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "SELECT department, COUNT(*) as emp_count, AVG(salary) as avg_salary\nFROM employees\nGROUP BY department\nHAVING COUNT(*) > 10 AND AVG(salary) > 60000;"
    },
    # Subqueries
    {
        "instruction": "Write a SQL query to find employees who earn more than the average salary.",
        "input": "Table: employees(id, name, salary)",
        "output": "SELECT name, salary\nFROM employees\nWHERE salary > (SELECT AVG(salary) FROM employees);"
    },
    {
        "instruction": "Write a SQL query to find the second highest salary.",
        "input": "Table: employees(id, name, salary)",
        "output": "SELECT MAX(salary) as second_highest\nFROM employees\nWHERE salary < (SELECT MAX(salary) FROM employees);"
    },
    # Window functions
    {
        "instruction": "Write a SQL query to rank employees by salary within each department.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "SELECT name, department, salary,\n       RANK() OVER (PARTITION BY department ORDER BY salary DESC) as salary_rank\nFROM employees;"
    },
    {
        "instruction": "Write a SQL query to calculate running total of sales by date.",
        "input": "Table: sales(id, sale_date, amount)",
        "output": "SELECT sale_date, amount,\n       SUM(amount) OVER (ORDER BY sale_date) as running_total\nFROM sales\nORDER BY sale_date;"
    },
    # Date functions
    {
        "instruction": "Write a SQL query to find orders placed in the last 30 days.",
        "input": "Table: orders(id, customer_id, amount, order_date)",
        "output": "SELECT * FROM orders\nWHERE order_date >= CURRENT_DATE - INTERVAL '30 days';"
    },
    {
        "instruction": "Write a SQL query to group sales by month and year.",
        "input": "Table: sales(id, sale_date, amount)",
        "output": "SELECT EXTRACT(YEAR FROM sale_date) as year,\n       EXTRACT(MONTH FROM sale_date) as month,\n       SUM(amount) as total_sales\nFROM sales\nGROUP BY EXTRACT(YEAR FROM sale_date), EXTRACT(MONTH FROM sale_date)\nORDER BY year, month;"
    },
    # String functions
    {
        "instruction": "Write a SQL query to find customers whose name starts with 'John'.",
        "input": "Table: customers(id, name, email)",
        "output": "SELECT * FROM customers\nWHERE name LIKE 'John%';"
    },
    {
        "instruction": "Write a SQL query to concatenate first_name and last_name with a space.",
        "input": "Table: employees(id, first_name, last_name)",
        "output": "SELECT CONCAT(first_name, ' ', last_name) as full_name\nFROM employees;"
    },
    # CASE statements
    {
        "instruction": "Write a SQL query to categorize employees by salary range (Low, Medium, High).",
        "input": "Table: employees(id, name, salary)",
        "output": "SELECT name, salary,\n       CASE\n           WHEN salary < 40000 THEN 'Low'\n           WHEN salary BETWEEN 40000 AND 80000 THEN 'Medium'\n           ELSE 'High'\n       END as salary_category\nFROM employees;"
    },
    # CTE (Common Table Expressions)
    {
        "instruction": "Write a SQL query using CTE to find employees with above-average salary in their department.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "WITH dept_avg AS (\n    SELECT department, AVG(salary) as avg_salary\n    FROM employees\n    GROUP BY department\n)\nSELECT e.name, e.salary, e.department, d.avg_salary\nFROM employees e\nJOIN dept_avg d ON e.department = d.department\nWHERE e.salary > d.avg_salary;"
    },
    # INSERT, UPDATE, DELETE
    {
        "instruction": "Write a SQL query to insert a new employee.",
        "input": "Table: employees(id, name, salary, department). Values: id=101, name='John Doe', salary=55000, department='Engineering'",
        "output": "INSERT INTO employees (id, name, salary, department)\nVALUES (101, 'John Doe', 55000, 'Engineering');"
    },
    {
        "instruction": "Write a SQL query to give all employees in Engineering department a 10% raise.",
        "input": "Table: employees(id, name, salary, department)",
        "output": "UPDATE employees\nSET salary = salary * 1.10\nWHERE department = 'Engineering';"
    },
    # Index and optimization
    {
        "instruction": "Write a SQL statement to create an index on the email column of customers table.",
        "input": "Table: customers(id, name, email)",
        "output": "CREATE INDEX idx_customers_email ON customers(email);"
    },
    {
        "instruction": "Explain how to optimize this slow query.",
        "input": "Query: SELECT * FROM orders WHERE YEAR(order_date) = 2024",
        "output": "The query is slow because YEAR(order_date) prevents index usage. Optimize by:\n\n1. Use a range condition instead:\nSELECT * FROM orders\nWHERE order_date >= '2024-01-01' AND order_date < '2025-01-01';\n\n2. Create an index on order_date:\nCREATE INDEX idx_orders_date ON orders(order_date);\n\nThis allows the database to use the index for filtering."
    },
    # More complex scenarios
    {
        "instruction": "Write a SQL query to find products that have never been ordered.",
        "input": "Tables: products(id, name, price), order_items(id, order_id, product_id, quantity)",
        "output": "SELECT p.id, p.name\nFROM products p\nWHERE NOT EXISTS (\n    SELECT 1 FROM order_items oi WHERE oi.product_id = p.id\n);"
    },
    {
        "instruction": "Write a SQL query to find the most popular product in each category.",
        "input": "Tables: products(id, name, category), order_items(id, product_id, quantity)",
        "output": "WITH product_sales AS (\n    SELECT p.category, p.name, SUM(oi.quantity) as total_sold,\n           ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY SUM(oi.quantity) DESC) as rn\n    FROM products p\n    JOIN order_items oi ON p.id = oi.product_id\n    GROUP BY p.category, p.name\n)\nSELECT category, name, total_sold\nFROM product_sales\nWHERE rn = 1;"
    },
    {
        "instruction": "Write a SQL query to calculate month-over-month revenue growth percentage.",
        "input": "Table: orders(id, order_date, amount)",
        "output": "WITH monthly_revenue AS (\n    SELECT DATE_TRUNC('month', order_date) as month,\n           SUM(amount) as revenue\n    FROM orders\n    GROUP BY DATE_TRUNC('month', order_date)\n)\nSELECT month, revenue,\n       LAG(revenue) OVER (ORDER BY month) as prev_month_revenue,\n       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0 /\n             NULLIF(LAG(revenue) OVER (ORDER BY month), 0), 2) as growth_pct\nFROM monthly_revenue\nORDER BY month;"
    }
]

print(f"Training examples: {len(SQL_TRAINING_DATA)}")

In [ ]:
# Format data for training
def format_prompt(example):
    """Format example into Llama 3 chat format"""
    instruction = example['instruction']
    input_text = example.get('input', '')
    output = example['output']
    
    if input_text:
        user_message = f"{instruction}\n\nContext:\n{input_text}"
    else:
        user_message = instruction
    
    # Llama 3 chat format
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL developer. Write clear, efficient SQL queries based on the user's requirements.<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""
    
    return {"text": text}

# Create dataset
formatted_data = [format_prompt(ex) for ex in SQL_TRAINING_DATA]
dataset = Dataset.from_list(formatted_data)

# Split into train/eval
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")
print(f"\nSample formatted prompt:\n{'-'*50}")
print(train_dataset[0]['text'][:500] + "...")

## 4. Load Model with 4-bit Quantization

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normalized float 4-bit
    bnb_4bit_compute_dtype=torch.float16, # Compute in fp16
    bnb_4bit_use_double_quant=True,       # Double quantization
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)

print(f"\nModel loaded!")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

## 5. Configure LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,                    # Rank
    lora_alpha=LORA_ALPHA,       # Scaling
    lora_dropout=LORA_DROPOUT,   # Dropout
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[             # Which layers to adapt
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 6. Test Base Model (Before Fine-tuning)

In [ ]:
# Test prompts for comparison
TEST_PROMPTS = [
    "Write a SQL query to find the top 3 products by revenue.",
    "Write a SQL query with a window function to calculate running total.",
    "Write a SQL query to find customers who ordered more than the average.",
]

def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate response from model"""
    full_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL developer. Write clear, efficient SQL queries.<|eot_id|><|start_header_id|>user<|end_header_id|>

{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract assistant response
    if "<|start_header_id|>assistant<|end_header_id|>" in response:
        response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.split("<|eot_id|>")[0].strip()
    return response

print("=" * 60)
print("BASE MODEL RESPONSES (Before Fine-tuning)")
print("=" * 60)

base_responses = []
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n--- Test {i} ---")
    print(f"Prompt: {prompt}")
    response = generate_response(model, tokenizer, prompt)
    base_responses.append(response)
    print(f"Response:\n{response}")

## 7. Train with QLoRA

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",  # Memory-efficient optimizer
    report_to="none",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting training...")
print(f"Total steps: {len(train_dataset) * EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

In [ ]:
# Train!
trainer.train()

In [ ]:
# Save the LoRA adapter
trainer.save_model(f"{OUTPUT_DIR}/final")
print(f"Model saved to {OUTPUT_DIR}/final")

## 8. Test Fine-tuned Model

In [ ]:
print("=" * 60)
print("FINE-TUNED MODEL RESPONSES (After Training)")
print("=" * 60)

finetuned_responses = []
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n--- Test {i} ---")
    print(f"Prompt: {prompt}")
    response = generate_response(model, tokenizer, prompt)
    finetuned_responses.append(response)
    print(f"Response:\n{response}")

## 9. Compare Base vs Fine-tuned

In [ ]:
print("=" * 80)
print("COMPARISON: Base vs Fine-tuned")
print("=" * 80)

for i, prompt in enumerate(TEST_PROMPTS):
    print(f"\n{'='*80}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*80}")
    print(f"\n📌 BASE MODEL:")
    print(f"{base_responses[i][:500]}")
    print(f"\n✅ FINE-TUNED MODEL:")
    print(f"{finetuned_responses[i][:500]}")

## 10. Save Results & Metrics

In [ ]:
# Get training metrics
metrics = trainer.state.log_history

# Extract loss values
train_losses = [m['loss'] for m in metrics if 'loss' in m]
eval_losses = [m['eval_loss'] for m in metrics if 'eval_loss' in m]

print("Training Summary")
print("=" * 40)
print(f"Initial loss: {train_losses[0]:.4f}")
print(f"Final loss: {train_losses[-1]:.4f}")
print(f"Loss reduction: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")

if eval_losses:
    print(f"Final eval loss: {eval_losses[-1]:.4f}")

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('QLoRA Fine-tuning Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.show()

## 11. Export & Next Steps

In [ ]:
# Save comparison results
results = {
    "model": MODEL_NAME,
    "adapter": NEW_MODEL_NAME,
    "training_examples": len(train_dataset),
    "epochs": EPOCHS,
    "lora_rank": LORA_R,
    "initial_loss": train_losses[0],
    "final_loss": train_losses[-1],
    "comparisons": [
        {"prompt": p, "base": b, "finetuned": f}
        for p, b, f in zip(TEST_PROMPTS, base_responses, finetuned_responses)
    ]
}

import json
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR}/results.json")

In [ ]:
# Download the adapter for local use
from google.colab import files

# Zip the adapter
!zip -r qlora_adapter.zip {OUTPUT_DIR}/final

# Download
files.download('qlora_adapter.zip')
print("\nAdapter downloaded! You can use this with Ollama or HuggingFace.")

## Summary

### What We Accomplished
1. ✅ Loaded Llama 3.1 8B with 4-bit quantization
2. ✅ Configured LoRA adapters (only ~0.1% params trainable)
3. ✅ Fine-tuned on SQL Q&A dataset
4. ✅ Compared base vs fine-tuned responses
5. ✅ Saved adapter for deployment

### Key Metrics
- **Memory used**: ~5GB (vs 32GB for full fine-tuning)
- **Training time**: ~45 minutes on T4
- **Adapter size**: ~4MB
- **Loss reduction**: See above

### Next Steps
- Deploy adapter with Ollama or vLLM
- Expand training dataset
- Experiment with different LoRA ranks
- Add more evaluation metrics